# 3 Gaussian Process and Bayesian Optimization (GP and BO)

## Keywords:

- GP: Variance, Covariance functions (Kernels), GP regression problem (Kriging)

- BO: objective function, prior, posterior distribution, sequential sampling, Acquisition functions

## Reference:

GP:

- Gaussian Processes for Machine Learning (http://gaussianprocess.org/gpml/chapters/RW.pdf)

- https://scikit-learn.org/stable/modules/gaussian_process.html

BO:

- https://www.youtube.com/watch?v=M-NTkxfd7-8
- Bayesian Optimization and Data Science by Francesco ArchettiAntonio Candelieri
- https://scikit-optimize.github.io/stable/auto_examples/bayesian-optimization.html
- https://modal-python.readthedocs.io/en/latest/content/examples/bayesian_optimization.html

We are going to use the modAL and Scikit learn package to carry out a simple 1-d toy problem. modAL provides nessasarry BO engines and Scikit learn the Gaussian regressor

In [ ]:
! pip install git+https://github.com/modAL-python/modAL.git

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from functools import partial
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF
from modAL.models import BayesianOptimizer
from modAL.acquisition import optimizer_EI, max_EI,max_UCB,PI
from ipywidgets.widgets import interact,interactive
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

plt.rcParams.update({'font.size': 12})
plt.rcParams["figure.figsize"] = (15,8)
plt.show()

## A simple 2D sample

In [ ]:
# generating the data
X = np.linspace(0, 20, 1000).reshape(-1, 1)
y = np.sin(X)/2 - ((10 - X)**2)/50 + 2

with plt.style.context('seaborn-v0_8-white'):
    plt.figure(figsize=(10, 5))
    plt.plot(X, y, c='k', linewidth=6)
    plt.title('The function to be optimized')
    plt.show()

In [ ]:
# assembling initial training set
X_initial, y_initial = X[250].reshape(1, -1), y[250].reshape(1, -1)

# defining the kernel for the Gaussian process
kernel = Matern(length_scale=1.0)
regressor = GaussianProcessRegressor(kernel=kernel)

In [ ]:
# initializing the optimizer
optimizer = BayesianOptimizer(estimator=regressor,
    X_training=X_initial,
    y_training=y_initial,
    query_strategy=max_EI)

# Bayesian optimization

for n_query in range(8):
    query_idx, query_inst = optimizer.query(X)
    optimizer.teach(X[query_idx].reshape(1, -1), y[query_idx].reshape(1, -1))
    PII=optimizer_EI(optimizer, X)

#Using expected improvement, the first five queries are the following.

    y_pred, y_std = optimizer.predict(X, return_std=True)
    y_pred, y_std = y_pred.ravel(), y_std.ravel()
    X_max, y_max = optimizer.get_max()

    with plt.style.context('seaborn-v0_8-white'):
        plt.figure(figsize=(30, 20))
        plt.subplot(2, 1, 1)

        if n_query == 0:
          plt.scatter(X_initial, y_initial, c='b', s=200, label='Initial')

        plt.scatter(optimizer.X_training, optimizer.y_training, c='k', s=50, label='Queried')
        plt.scatter(X_max, y_max, s=100, c='r', label='Current optimum')
        plt.plot(X.ravel(), y, "--y", linewidth=2, label='Function')
        plt.plot(X.ravel(), y_pred, label='GP regressor')
        plt.fill_between(X.ravel(), y_pred - y_std, y_pred + y_std, alpha=0.5)
        plt.title(f' {n_query+1} query of Bayesian optimization')
        plt.ylabel('y')
        plt.legend()
        plt.show()

        plt.figure(figsize=(30, 20))
        plt.subplot(2, 1, 2)
        plt.plot(X.ravel(),PII)
        plt.xlabel('X')
        plt.ylabel('PI')
        plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0.1))
        plt.show()

### Explanation of the Bayesian Optimization Loop

This loop performs **8 iterations** of sequential sampling to optimize the unknown target function:

1. `query_idx, query_inst = optimizer.query(X)`:
   - **Querying**: The optimizer uses its acquisition function (specifically `max_EI` here, which stands for **Maximum Expected Improvement**) to scan all candidate points in `X` and find the single point that is most promising to evaluate next. It returns the index (`query_idx`) and the value of that point (`query_inst`).

2. `optimizer.teach(X[query_idx].reshape(1, -1), y[query_idx].reshape(1, -1))`:
   - **Teaching**: The optimizer evaluates the true function value ($y$) at the queried location and adds this new training sample back into its history. It then retrains the underlying Gaussian Process (GP) regressor with the updated dataset.

3. `PII = optimizer_EI(optimizer, X)`:
   - **Acquisition Calculation**: This calculates the Expected Improvement (EI) values for all candidate points across the entire search space `X` using the newly trained GP. This is saved to `PII` so it can be plotted in the subplot below to visualize how the acquisition landscape changes.

4. `y_pred, y_std = optimizer.predict(X, return_std=True)`:
   - **Prediction**: The updated GP predicts the mean values (`y_pred`) and the associated uncertainty/standard deviation (`y_std`) for all points in `X`. This is used to draw the mean regression line and the shaded uncertainty interval ($
 y \pm \sigma$).

5. `X_max, y_max = optimizer.get_max()`:
   - **Best Location**: This retrieves the coordinates (`X_max`) and corresponding evaluation (`y_max`) of the best overall sample found in the training history so far.

In [ ]:
X_1d = np.linspace(0, 20, 1000).reshape(-1, 1)
y_1d = np.sin(X_1d)/2 - ((10 - X_1d)**2)/50 + 2
X_initial_1d, y_initial_1d = X_1d[250].reshape(1, -1), y_1d[250].reshape(1, -1)

# Dedicated 1D interactive function to prevent conflicts with 3D model globals
def interactive_1d(n_query=0):
    # Re-initialize local GP and Optimizer matching the step
    regressor_1d = GaussianProcessRegressor(kernel=Matern(length_scale=1.0))
    optimizer_1d = BayesianOptimizer(
        estimator=regressor_1d,
        X_training=X_initial_1d,
        y_training=y_initial_1d,
        query_strategy=max_EI
    )

    for _ in range(n_query):
        query_idx, query_inst = optimizer_1d.query(X_1d)
        optimizer_1d.teach(X_1d[query_idx].reshape(1, -1), y_1d[query_idx].reshape(1, -1))

    y_pred, y_std = optimizer_1d.predict(X_1d, return_std=True)
    y_pred, y_std = y_pred.ravel(), y_std.ravel()
    X_max, y_max = optimizer_1d.get_max()

    with plt.style.context('seaborn-v0_8-white'):
        plt.figure(figsize=(10, 5))
        plt.scatter(optimizer_1d.X_training, optimizer_1d.y_training, c='k', s=50, label='Queried')
        plt.scatter(X_max, y_max, s=100, c='r', label='Current optimum')
        plt.plot(X_1d.ravel(), y_1d, "--y", linewidth=2, label='Function')
        plt.plot(X_1d.ravel(), y_pred, label='GP regressor')
        plt.fill_between(X_1d.ravel(), y_pred - y_std, y_pred + y_std, alpha=0.5)
        plt.title('{} queries of Bayesian optimization'.format(n_query))
        plt.legend()
        plt.show()

interact(interactive_1d, n_query=(0,11,1))

In [ ]:
X.ravel().shape

## A 3D Sample

Now we increase the dimensionality of the sample and check out how BO does when two features are included.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib.ticker import LinearLocator, FormatStrFormatter
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

#2-Dimensional Example

def obj_func_2d(x,y):
    return((x**2 + y**2)*(np.sin(x)**2 - np.cos(y))+300)


# Make data.
X = np.arange(5, 10, 0.1)
Y = np.arange(5, 10, 0.1)
XX, YY = np.meshgrid(X, Y)
Z = obj_func_2d(XX,YY)

#fig = plt.figure()
#ax = fig.add_subplot(1, 1, 1)
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
# Plot the surface.

surf = ax.plot_trisurf(XX.flatten(), YY.flatten(), Z.flatten(), cmap=cm.coolwarm,
                       linewidth=0, antialiased=False,alpha=0.6)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel(r"f(x,y)")


# # Add a color bar which maps values to colors.
# fig.colorbar(surf, shrink=0.5, aspect=5)
# fig = plt.figure(figsize=plt.figaspect(0.3))
# fig.suptitle('Plots of our objective function')
# # First subplot
# fig = plt.figure()
# ax = fig.add_subplot(1, 1, 1)
# ax.contour(XX,YY,Z)

plt.show()




x,y can be any input feature descriptors and f(x,y) any output target

Now we do the same procedure for 3D

In [ ]:
#@title
## shape the data
X_s = np.vstack((XX.flatten(), YY.flatten()))
X_s=X_s.T
print(np.shape(X_s))

## scaling the data
SS_x = StandardScaler()
SS_y = StandardScaler()

X_t = SS_x.fit_transform(X_s)
y = Z.flatten()
y = SS_y.fit_transform(y.reshape(-1, 1))

print(np.shape(y))

#assembling initial training set
X_initial, y_initial = X_t[90,:].reshape(1, -1), y[90].reshape(1, -1)

# defining the kernel for the Gaussian process
kernel = Matern(length_scale=1.0)
regressor = GaussianProcessRegressor(kernel=kernel)

# initializing the optimizer
optimizer = BayesianOptimizer(
    estimator=regressor,
    X_training=X_initial,
    y_training=y_initial,
    query_strategy=max_EI
)

# query history
query_list=[]
x_1=[]
x_2=[]
z_=[]

# Prepare 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(projection='3d')

surf = ax.plot_trisurf(XX.flatten(), YY.flatten(), Z.flatten(), cmap=cm.coolwarm,
                       linewidth=0, antialiased=False, alpha=0.6)

# Bayesian optimization
for n_query in range(10):
    query_idx, query_inst = optimizer.query(X_t)
    query_list.append(query_idx)
    optimizer.teach(X_t[query_idx,:].reshape(1, -1), y[query_idx].reshape(1, -1))

    y_pred, y_std = optimizer.predict(X_t, return_std=True)
    y_pred, y_std = y_pred.ravel(), y_std.ravel()
    X_max, y_max = optimizer.get_max()

    x1 = float(SS_x.inverse_transform(X_t[query_idx,:].reshape(1, -1))[0][0])
    x2 = float(SS_x.inverse_transform(X_t[query_idx,:].reshape(1, -1))[0][1])
    z = float(SS_y.inverse_transform(y[query_idx][0].reshape(-1, 1))[0][0])

    x_1.append(x1)
    x_2.append(x2)
    z_.append(z)
    ax.scatter(x1, x2, z, c='k', s=50)
    ax.text(x1, x2, z, "No. {:}  ".format(n_query), zdir=(0, 0, 0))

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel(r"f(x,y)")

# Plot the path of BO queries
ax.plot(x_1, x_2, z_, 'r-->', alpha=0.8, linewidth=2)

##########get the max value######
X_max, y_max_in_queried = optimizer.get_max()
print(np.shape(X_max))

X_max_pred = SS_x.inverse_transform(X_max.reshape(1, -1))
y_max_data = SS_y.inverse_transform(y_max_in_queried.reshape(1, -1))

print("Best points after n queries are:", X_max_pred, y_max_data)
plt.show()

In [ ]:
## check this is indeed the maxium in our dataset

data = np.stack((XX.flatten(), YY.flatten(), Z.flatten()), axis = 1)
data = pd.DataFrame(data)
data

In [ ]:
np.max(data[2])